# ✏️ TOONIFY — Sketch LoRA Model Training Notebook

Train a custom Stable Diffusion 1.5 LoRA model using your **400-sketch dataset** (`20_abhi_sketch_style person`).

- **Base Model**: `runwayml/stable-diffusion-v1-5`
- **Trigger Word**: `abhi_sketch_style`
- **Dataset**: PNG images + matching `.txt` caption files
- **Output**: `sketch_lora.safetensors`

## Step 1: Install Dependencies
Run this cell on Google Colab (with T4 GPU enabled).

In [ ]:
!nvidia-smi
!pip install -q diffusers transformers accelerate peft bitsandbytes safetensors torchvision xformers

## Step 2: Mount Google Drive & Prepare Dataset
Upload `sketch_dataset.zip` to Google Drive or directly to Colab.

In [ ]:
import os
import zipfile
from google.colab import drive

if os.path.exists('/content/drive'):
    print('Google Drive already mounted.')
else:
    drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/sketch_dataset.zip'
if not os.path.exists(zip_path):
    zip_path = '/content/sketch_dataset.zip'

dataset_dir = '/content/dataset'
os.makedirs(dataset_dir, exist_ok=True)

if os.path.exists(zip_path):
    print(f'Unzipping dataset from {zip_path}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
    print('Dataset extracted successfully!')
else:
    print('⚠️ Please upload sketch_dataset.zip to Google Drive or /content/ directory.')

## Step 3: Verify Dataset Pairs (.png + .txt)

In [ ]:
import glob

image_files = sorted(glob.glob('/content/dataset/**/*.png', recursive=True) + glob.glob('/content/dataset/**/*.jpg', recursive=True))
print(f'Found {len(image_files)} image files.')

valid_pairs = []
for img in image_files:
    txt_file = os.path.splitext(img)[0] + '.txt'
    if os.path.exists(txt_file):
        with open(txt_file, 'r', encoding='utf-8') as f:
            caption = f.read().strip()
        valid_pairs.append((img, caption))

print(f'✅ Found {len(valid_pairs)} complete image + caption pairs.')
if valid_pairs:
    print('Sample image:', valid_pairs[0][0])
    print('Sample caption:', valid_pairs[0][1])

## Step 4: Train Sketch LoRA Model

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

MODEL_ID = 'runwayml/stable-diffusion-v1-5'
OUTPUT_DIR = '/content/output_lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder='text_encoder').to('cuda', dtype=torch.float16)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder='vae').to('cuda', dtype=torch.float16)
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder='unet').to('cuda', dtype=torch.float16)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')

vae.requires_grad_(False)
text_encoder.requires_grad_(False)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=['to_k', 'to_q', 'to_v', 'to_out.0'],
    lora_dropout=0.05,
    bias='none',
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

class SketchDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
        self.transform = transforms.Compose([
            transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        img_path, caption = self.pairs[idx]
        image = Image.open(img_path).convert('RGB')
        tensor = self.transform(image)
        tokens = tokenizer(
            caption,
            padding='max_length',
            max_length=tokenizer.model_max_length,
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze(0)
        return {'image': tensor, 'tokens': tokens}

dataset = SketchDataset(valid_pairs)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)
num_epochs = 5
print(f'Starting training for {num_epochs} epochs ({len(dataloader) * num_epochs} steps)...')

unet.train()
for epoch in range(num_epochs):
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{num_epochs}')
    for batch in progress_bar:
        images = batch['image'].to('cuda', dtype=torch.float16)
        tokens = batch['tokens'].to('cuda')

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * 0.18215
            encoder_hidden_states = text_encoder(tokens)[0]

        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device='cuda').long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = torch.nn.functional.mse_loss(noise_pred.float(), noise.float(), reduction='mean')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

unet.save_pretrained(OUTPUT_DIR)
print(f'✅ Training complete! Saved LoRA model to {OUTPUT_DIR}')


## Step 5: Test Model Generation in Colab

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None
).to('cuda')

pipe.unet.load_attn_procs(OUTPUT_DIR)

prompt = 'portrait of a young woman, abhi_sketch_style, pencil sketch, detailed crosshatching, graphite art'
image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]
image.save('/content/test_sketch_output.png')
display(image)
